# giving ans in a structured way

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY");

In [1]:
from langchain.chat_models import init_chat_model
model  = init_chat_model(
    model="groq:qwen/qwen3-32b"
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x11b4c2660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11b4c3380>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import Field
from pydantic import BaseModel
class Movie(BaseModel):
    name:str=Field(...,description="This is the name of the movie")
    director:str=Field(...,description="This is the name of the director of the movie")
    rating:float=Field(...,le=10.0,description="This is the rating of the movie out of 10")
    year:int=Field(...,description="The year in which movie was released")

model_with_structure = model.with_structured_output(Movie)

response = model_with_structure.invoke("tell me about movie 3 idiots")
response

Movie(name='3 Idiots', director='Rajkumar Hirani', rating=8.4)

In [14]:
from pydantic import Field
from pydantic import BaseModel
class Movie(BaseModel):
    name:str=Field(...,description="This is the name of the movie")
    director:str=Field(...,description="This is the name of the director of the movie")
    rating:float=Field(...,le=10.0,description="This is the rating of the movie out of 10")
    year:int=Field(...,description="The year in which movie was released")

model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("tell me about movie 3 idiots")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie "3 Idiots." Let me see what I need to do here. They want information about that movie. Looking at the tools provided, there\'s a function called Movie that requires parameters like director, name, rating, and year. So I need to use that function to retrieve the details.\n\nFirst, I need to check if I have all the required parameters. The movie name is given as "3 Idiots." I should confirm the director\'s name. I think it\'s Rajkumar Hirani. The rating is probably around 8 or 9 out of 10. The release year was 2009, I believe. Let me verify that. Yes, "3 Idiots" was released in 2009, directed by Rajkumar Hirani, and it has a high rating, maybe 8.4 or something similar. \n\nI need to make sure all required fields are included. The function requires name, director, rating, and year. So I\'ll structure the tool call with those parameters. Let me double-check the maximum rating all

# NESTED

In [16]:
from typing import Optional
from typing import List
# NESTED

class Actor(BaseModel):
    name:str
    age:int

class Movie1(BaseModel):
    name:str=Field(...,description="This is the name of the movie")
    cast : List[Actor]=Field(description="This is the actors who acted in the movie")
    genre:List[str]
    rating:float=Field(...,le=10.0,description="This is the rating of the movie out of 10")
    year:int=Field(...,description="The year in which movie was released")
    budget:Optional[float]=Field(description="The budget of the movie")
    

model_with_structure = model.with_structured_output(Movie1)
model_with_structure.invoke("Tell me about movie 3 idiots")

Movie1(name='3 Idiots', cast=[Actor(name='Aamir Khan', age=45), Actor(name='Rajkumar Rao', age=29), Actor(name='Madhavan', age=38)], genre=['Comedy', 'Drama'], rating=8.4, year=2009, budget=10000000.0)

# typeDict
## no runtime validation like pydantic


In [17]:
from typing import Annotated
from typing import TypedDict
class Movie(TypedDict):
    title:Annotated[str,...,"the title of the movie or the name of the movie"]
    year:Annotated[int,...,"the year in which movie was released"]
    rating:Annotated[float,...,"the rating of the movie out of 10"]

model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("tell me about movie interstellar")


{'rating': 8.6, 'title': 'Interstellar', 'year': 2014}

# what the model supports

In [18]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# Data class

In [19]:
from dataclasses import dataclass
@dataclass
class ContactInfo:
    name:str
    phone:int
    address:str


model_with_structure1 = model.with_structured_output(ContactInfo)
model_with_structure1.invoke("extract contact info from vedish street4 downtown 819181991")

{'address': 'street4 downtown', 'name': 'vedish', 'phone': 819181991}